# ChemSynonymizer Colab Pipeline

Run this notebook top to bottom on Colab. Cell 1 installs dependencies. Cell 2 mounts Google Drive. After that, set the repo and CSV paths, run baselines, make plots, and generate analysis tables for the results section.

In [ ]:
# Cell 1: install runtime dependencies.
# Keep Colab's pinned pandas/numpy stack. Do not use `-U pandas numpy` here.
# Colab currently expects pandas==2.2.2, and RAPIDS/numba packages expect numpy<2.1.
!pip -q install 'pandas==2.2.2' 'numpy>=1.26,<2.1'
!pip -q install scipy scikit-learn tqdm PyYAML matplotlib joblib transformers accelerate sentencepiece hf_transfer
!pip -q install faiss-gpu-cu12 || pip -q install faiss-cpu

# If this cell had previously installed incompatible pandas/numpy versions,
# restart the Colab runtime after this cell, then continue from Cell 2.

In [ ]:
# Cell 2: mount Google Drive.
from google.colab import drive
drive.mount('/content/drive')

## Configure Paths

Edit `REPO_DIR` and `CSV_PATH` below. A common setup is to place the `chem_synonymizer` folder somewhere in `MyDrive` and keep the CSV beside it or in a data folder.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import shutil

# EDIT THESE TWO PATHS.
REPO_DIR = Path('/content/drive/MyDrive/ChemSynonymizer/chem_synonymizer')
CSV_PATH = Path('/content/drive/MyDrive/ChemSynonymizer/fixedGood.csv')

assert REPO_DIR.exists(), f'Repo folder not found: {REPO_DIR}'
assert (REPO_DIR / 'scripts').exists(), f'Scripts folder not found under: {REPO_DIR}'
assert CSV_PATH.exists(), f'CSV not found: {CSV_PATH}'

# Fast active cache: use local scratch for huge embedding/index files.
# Results stay in REPO_DIR/results on Drive; cache can be rebuilt if the runtime is lost.
DRIVE_CACHE_ROOT = REPO_DIR / 'cache'
SCRATCH_CANDIDATES = [Path('/content/local-scratch'), Path('/local-scratch'), Path('/content')]
SCRATCH_ROOT = next((p for p in SCRATCH_CANDIDATES if p.exists()), Path('/content'))
CACHE_ROOT = SCRATCH_ROOT / 'chem_synonymizer_cache'
HF_CACHE = CACHE_ROOT / 'huggingface'
TORCH_CACHE = CACHE_ROOT / 'torch'
HF_CACHE.mkdir(parents=True, exist_ok=True)
TORCH_CACHE.mkdir(parents=True, exist_ok=True)
DRIVE_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_CACHE)
os.environ['HF_HUB_CACHE'] = str(HF_CACHE / 'hub')
os.environ['TRANSFORMERS_CACHE'] = str(HF_CACHE / 'transformers')
os.environ['TORCH_HOME'] = str(TORCH_CACHE)
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM'] = 'true'
os.environ['OMP_NUM_THREADS'] = str(max(1, min(8, os.cpu_count() or 1)))

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / 'src'))
print('Working directory:', Path.cwd())
print('CSV:', CSV_PATH)
print('Active scratch cache:', CACHE_ROOT)
print('Drive cache folder for optional backup:', DRIVE_CACHE_ROOT)
for label, path in [('scratch', SCRATCH_ROOT), ('drive repo', REPO_DIR)]:
    usage = shutil.disk_usage(path)
    print(f'{label} free GB:', round(usage.free / 1e9, 1), 'of', round(usage.total / 1e9, 1))

In [ ]:
import torch
import faiss

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
print('faiss:', faiss.__version__)
print('faiss gpu resources:', hasattr(faiss, 'StandardGpuResources'))

## Experiment Settings

For the first Colab test, set `LIMIT_ROWS` to something small like `5000`. For the real run, set it to `None`. The default semantic models below cover the requested pretrained baselines.

In [ ]:
EVAL_FRAC = 0.25
LIMIT_ROWS = None  # Example smoke test: 5000
RUN_LEXICAL_BASELINE = True
RUN_SEMANTIC_BASELINES = True
SEMANTIC_MODELS = ['e5_base', 'e5_large', 'bge_base', 'biomedbert', 'sapbert']
RUN_ROBUSTNESS = True  # Runs robustness for every semantic model in the main loop.
RUN_LEXICAL_ROBUSTNESS = False
ROBUSTNESS_SEMANTIC_MODEL = 'sapbert'  # Only used if you run the separate legacy robustness cell.
RUN_SAPBERT_FINETUNING = False  # Set True for the fine-tuning phase after pretrained baselines are done.
SAPBERT_FINETUNE_OBJECTIVES = ['contrastive', 'triplet']
SAPBERT_INCLUDE_HARD_NEGATIVE_RUNS = False  # Set True to also run contrastive_hard and triplet_hard.
SAPBERT_FINETUNE_STEPS_PER_EPOCH = 2000
SAPBERT_FINETUNE_BATCH_SIZE = 128
INDEX_TYPE = 'ivf_flat'
FAISS_DEVICE = 'cuda'  # 80GB GPU path: embeddings and FAISS both use CUDA.
RESTORE_CACHE_FROM_DRIVE = True  # Before a model run, copy its saved cache from Drive back to scratch if available.
BACKUP_CACHE_TO_DRIVE = True  # After each model finishes, save that model's scratch cache to Drive.
BACKUP_FAISS_INDEXES_TO_DRIVE = False  # Keep False to upload embeddings but skip huge IVF-flat index files.
KEEP_FAISS_INDEXES = False  # After backup, delete ~40-60GB/model FAISS indexes from scratch.
from collections import deque
from concurrent.futures import ThreadPoolExecutor
CACHE_COPY_WORKERS = 2

def copy_changed_tree(src, dst, skip_parts=None):
    src = Path(src)
    dst = Path(dst)
    skip_parts = set(skip_parts or [])
    if not src.exists():
        return 0
    jobs = []
    for path in src.rglob('*'):
        rel = path.relative_to(src)
        if any(part in skip_parts for part in rel.parts):
            continue
        target = dst / rel
        if path.is_dir():
            target.mkdir(parents=True, exist_ok=True)
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        should_copy = not target.exists()
        if not should_copy:
            s = path.stat()
            t = target.stat()
            should_copy = (s.st_size != t.st_size) or (s.st_mtime > t.st_mtime + 1)
        if should_copy:
            jobs.append((path, target))
    if not jobs:
        return 0
    with ThreadPoolExecutor(max_workers=CACHE_COPY_WORKERS) as pool:
        list(pool.map(lambda pair: shutil.copy2(pair[0], pair[1]), jobs))
    return len(jobs)

def semantic_relative_path(path):
    path = Path(path)
    parts = path.parts
    if 'semantic' not in parts:
        return None
    i = parts.index('semantic')
    return Path(*parts[i:])

def restore_run_cache_from_drive(run_name):
    if not RESTORE_CACHE_FROM_DRIVE:
        return
    info_path = REPO_DIR / 'results' / run_name / 'cache_info.json'
    if not info_path.exists():
        return
    import json
    info = json.loads(info_path.read_text())
    restored = 0
    for key in ['embedding_cache_dir']:
        rel = semantic_relative_path(info.get(key, ''))
        if rel is None:
            continue
        src = DRIVE_CACHE_ROOT / rel
        dst = CACHE_ROOT / rel
        restored += copy_changed_tree(src, dst)
    if restored:
        print(f'Restored {restored} changed cache files for {run_name} from Drive to scratch.')

def backup_run_cache_to_drive(run_name):
    if not BACKUP_CACHE_TO_DRIVE:
        return
    info_path = REPO_DIR / 'results' / run_name / 'cache_info.json'
    if not info_path.exists():
        print('No cache_info.json found for backup:', run_name)
        return
    import json
    info = json.loads(info_path.read_text())
    rel = semantic_relative_path(info.get('embedding_cache_dir', ''))
    if rel is None:
        print('No semantic embedding cache path found for backup:', run_name)
        return
    src = CACHE_ROOT / rel
    dst = DRIVE_CACHE_ROOT / rel
    skip_parts = [] if BACKUP_FAISS_INDEXES_TO_DRIVE else ['indexes']
    copied = copy_changed_tree(src, dst, skip_parts=skip_parts)
    skipped = '' if BACKUP_FAISS_INDEXES_TO_DRIVE else ' (skipped FAISS indexes)'
    print(f'Backed up {copied} changed cache files for {run_name} to Drive{skipped}: {dst}')

def drop_index_cache_for_run(run_name):
    if KEEP_FAISS_INDEXES:
        return
    info_path = REPO_DIR / 'results' / run_name / 'cache_info.json'
    if not info_path.exists():
        return
    import json
    info = json.loads(info_path.read_text())
    index_dir = info.get('index_cache_dir')
    if not index_dir:
        return
    index_dir = Path(index_dir)
    if index_dir.exists() and str(index_dir).startswith(str(CACHE_ROOT)):
        shutil.rmtree(index_dir)
        print('Deleted FAISS index cache to save scratch space:', index_dir)

def run(cmd):
    cmd = list(map(str, cmd))
    print('\n$ ' + ' '.join(cmd))
    last_lines = deque(maxlen=80)
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='')
        last_lines.append(line.rstrip())
    returncode = process.wait()
    if returncode != 0:
        tail = '\n'.join(last_lines)
        raise RuntimeError(f'Command failed with exit code {returncode}. Last output lines:\n{tail}')
    return returncode

common_args = ['--csv-path', CSV_PATH, '--eval-frac', EVAL_FRAC, '--set', f'project.cache_dir={CACHE_ROOT}']
if LIMIT_ROWS is not None:
    common_args += ['--limit-rows', LIMIT_ROWS]

print('Models:', SEMANTIC_MODELS)
print('Common args:', common_args)

## Run Baselines

This runs the lexical baseline first, then each semantic model. If `RUN_ROBUSTNESS=True`, each semantic model also runs its robustness evaluation before the next model starts. Outputs are saved in stable folders under `results/`, one folder per baseline/model label.

In [ ]:
python = sys.executable

if RUN_LEXICAL_BASELINE:
    run([python, 'scripts/run_lexical_baseline.py', '--run-name', 'lexical', *common_args])
else:
    print('Skipping lexical baseline.')

if RUN_SEMANTIC_BASELINES:
    for model_key in SEMANTIC_MODELS:
        restore_run_cache_from_drive(model_key)
        semantic_cmd = [
            python,
            'scripts/run_semantic_baseline.py',
            '--run-name', model_key,
            '--model-key', model_key,
            '--device', 'cuda',
            '--faiss-device', FAISS_DEVICE,
            '--index-type', INDEX_TYPE,
            *common_args,
        ]
        if RUN_ROBUSTNESS:
            semantic_cmd.append('--run-robustness')
        run(semantic_cmd)
        backup_run_cache_to_drive(model_key)
        drop_index_cache_for_run(model_key)
else:
    print('Skipping pretrained semantic baselines.')

## Optional Separate Robustness

Semantic robustness is already handled inside the main model loop above. This optional cell is only for the lexical baseline or a one-off legacy semantic robustness run.

In [ ]:
if RUN_LEXICAL_ROBUSTNESS:
    run([python, 'scripts/run_robustness.py', '--run-name', 'robustness_lexical', '--baseline', 'lexical', *common_args])
else:
    print('Skipping lexical robustness.')

RUN_LEGACY_SINGLE_MODEL_ROBUSTNESS = False
if RUN_LEGACY_SINGLE_MODEL_ROBUSTNESS:
    restore_run_cache_from_drive(f'robustness_{ROBUSTNESS_SEMANTIC_MODEL}')
    run([
        python,
        'scripts/run_robustness.py',
        '--run-name', f'robustness_{ROBUSTNESS_SEMANTIC_MODEL}',
        '--baseline', 'semantic',
        '--model-key', ROBUSTNESS_SEMANTIC_MODEL,
        '--device', 'cuda',
        '--faiss-device', FAISS_DEVICE,
        '--index-type', INDEX_TYPE,
        *common_args,
    ])
    backup_run_cache_to_drive(f'robustness_{ROBUSTNESS_SEMANTIC_MODEL}')
    drop_index_cache_for_run(f'robustness_{ROBUSTNESS_SEMANTIC_MODEL}')
else:
    print('Skipping separate semantic robustness; main loop already covers all semantic models.')

## Fine-Tune SapBERT

This optional phase trains SapBERT on the synonym sets with supervised contrastive loss and triplet loss. Each tuned checkpoint is then evaluated with the same semantic retrieval and robustness pipeline as the pretrained models.

In [ ]:
def sapbert_finetune_runs():
    runs = []
    for objective in SAPBERT_FINETUNE_OBJECTIVES:
        runs.append({'run_name': f'sapbert_{objective}', 'objective': objective, 'hard': False})
        if SAPBERT_INCLUDE_HARD_NEGATIVE_RUNS:
            runs.append({'run_name': f'sapbert_{objective}_hard', 'objective': objective, 'hard': True})
    return runs

if RUN_SAPBERT_FINETUNING:
    for spec in sapbert_finetune_runs():
        run_name = spec['run_name']
        model_dir = REPO_DIR / 'results' / run_name / 'artifacts' / 'model'
        train_cmd = [
            python,
            'scripts/run_sapbert_finetuning.py',
            '--run-name', run_name,
            '--objective', spec['objective'],
            '--device', 'cuda',
            *common_args,
            '--set', f'finetune.steps_per_epoch={SAPBERT_FINETUNE_STEPS_PER_EPOCH}',
            '--set', f'finetune.batch_size={SAPBERT_FINETUNE_BATCH_SIZE}',
        ]
        if spec['hard']:
            train_cmd.append('--use-hard-negatives')
        run(train_cmd)

        restore_run_cache_from_drive(run_name)
        eval_cmd = [
            python,
            'scripts/run_semantic_baseline.py',
            '--run-name', run_name,
            '--model-key', run_name,
            '--model-name', model_dir,
            '--pooling', 'cls',
            '--device', 'cuda',
            '--faiss-device', FAISS_DEVICE,
            '--index-type', INDEX_TYPE,
            *common_args,
        ]
        if RUN_ROBUSTNESS:
            eval_cmd.append('--run-robustness')
        run(eval_cmd)
        backup_run_cache_to_drive(run_name)
        drop_index_cache_for_run(run_name)
else:
    print('Skipping SapBERT fine-tuning.')

## Generate Summary Plots

In [ ]:
run([python, 'scripts/make_plots.py'])

## Load Results

In [ ]:
import pandas as pd
from IPython.display import display, Image, Markdown
from chem_synonymizer.plots import load_metric_runs, load_bucket_metrics, load_robustness_metrics

RESULTS_DIR = REPO_DIR / 'results'
ANALYSIS_DIR = RESULTS_DIR / 'analysis_tables'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

metrics = load_metric_runs(RESULTS_DIR)
buckets = load_bucket_metrics(RESULTS_DIR)
robustness = load_robustness_metrics(RESULTS_DIR)

metric_cols = [c for c in ['baseline', 'model_key', 'n_queries', 'MRR', 'HitRate@1', 'HitRate@5', 'HitRate@10', 'Precision@1', 'Precision@5', 'Precision@10', 'RecallFrac@1', 'RecallFrac@5', 'RecallFrac@10'] if c in metrics.columns]
metrics_table = metrics[metric_cols].sort_values('MRR', ascending=False).reset_index(drop=True)
metrics_table.to_csv(ANALYSIS_DIR / 'baseline_metrics_table.csv', index=False)
display(metrics_table)

In [ ]:
if not buckets.empty:
    bucket_cols = [c for c in ['baseline', 'model_key', 'bucket', 'n_queries', 'MRR', 'HitRate@1', 'HitRate@5', 'HitRate@10', 'RecallFrac@10'] if c in buckets.columns]
    bucket_table = buckets[bucket_cols].sort_values(['model_key', 'bucket']).reset_index(drop=True)
    bucket_table.to_csv(ANALYSIS_DIR / 'bucket_metrics_table.csv', index=False)
    display(bucket_table)
else:
    print('No bucket metrics found.')

In [ ]:
if not robustness.empty:
    robust_cols = [c for c in ['baseline', 'model_key', 'corruption_level', 'n_queries', 'MRR', 'HitRate@1', 'HitRate@5', 'HitRate@10'] if c in robustness.columns]
    robustness_table = robustness[robust_cols].sort_values(['model_key', 'corruption_level']).reset_index(drop=True)
    robustness_table.to_csv(ANALYSIS_DIR / 'robustness_metrics_table.csv', index=False)
    display(robustness_table)
else:
    print('No robustness metrics found.')

## Show Plots

In [ ]:
plot_dir = RESULTS_DIR / 'summary_plots'
for plot_name in ['mrr_by_model.png', 'hitrate_by_model.png', 'bucket_mrr.png', 'robustness_mrr.png']:
    path = plot_dir / plot_name
    if path.exists():
        display(Markdown(f'### {plot_name}'))
        display(Image(filename=str(path)))

## Draft Results Text

This creates a short editable draft from the tables. Treat it as a starting point, not final prose.

In [ ]:
if metrics.empty:
    print('No metrics available yet.')
else:
    best = metrics_table.iloc[0]
    lexical_rows = metrics_table[metrics_table['baseline'] == 'lexical']
    lexical_text = ''
    if not lexical_rows.empty:
        lex = lexical_rows.iloc[0]
        lexical_text = f"The lexical char n-gram TF-IDF baseline reached MRR={lex['MRR']:.3f}, HitRate@1={lex.get('HitRate@1', float('nan')):.3f}, and HitRate@10={lex.get('HitRate@10', float('nan')):.3f}."

    draft = f"""## Results Draft

We evaluated chemical synonym retrieval on the non-singleton subset of the CSV, using each synonym surface form as a query and all other names as retrieval candidates. The main metrics were MRR, HitRate@k, Precision@k, and RecallFrac@k, with self-matches removed before scoring.

{lexical_text}

Among the evaluated systems, the best overall run was `{best['model_key']}` with MRR={best['MRR']:.3f}, HitRate@1={best.get('HitRate@1', float('nan')):.3f}, HitRate@5={best.get('HitRate@5', float('nan')):.3f}, and HitRate@10={best.get('HitRate@10', float('nan')):.3f} over {int(best['n_queries'])} queries.

Bucketed diagnostics by synonym-set size should be used to determine whether gains are concentrated in compounds with many aliases or persist for compounds with only two or three known names. Robustness curves should be used to describe how retrieval quality degrades under corrupted query names.
"""
    out_path = ANALYSIS_DIR / 'results_section_draft.md'
    out_path.write_text(draft, encoding='utf-8')
    display(Markdown(draft))
    print('Saved:', out_path)